# Study on Reflection pattern of agentic programming

## Prepare

In [1]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


In [2]:
with open('api-key/deepseek.txt', 'r') as f:
    llm = ChatOpenAI(
        base_url="https://api.deepseek.com",
        api_key=f.read().strip(),
        model="deepseek-v4-flash",
        temperature=0,
    )

# print(
#     StrOutputParser().invoke(
#         llm.invoke([("human", "Introduce yourself briefly")])
#     )
# )

## Predefined Prompts

In [7]:
# prompt of main task
task_prompt = """你的任务是创建一个名为 `calculate_factorial` 的 Python 函数，满足：
- 只接受一个整数参数 n
- 计算其阶乘 (n!)
- 包含清晰的 docstring, 说明函数功能
- 处理边界情况 (0 的阶乘为 1)
- 处理无效输入 (若输入为负数则抛出 ValueError)
"""

# success key
success_key = "CODE_IS_PERFECT"

# prompt of reflection
reflect_prompt = f"""你是一名资深软件工程师，精通 Python。
你的职责是对提供的 Python 代码进行细致代码审查。
请根据原始任务要求，严格评估代码。
检查是否有 bug、风格问题、遗漏边界情况及其他可改进之处。
若代码完美且满足所有要求，仅回复 '{success_key}'。
否则，请以项目符号列表形式给出批判意见。
"""

## Reflection loop

In [ ]:
# initialize loop
max_step = 5
current_code = ""
message_history = [HumanMessage(content=task_prompt)]

In [5]:
for i in range(max_step):
    print("="*8 + f"反思循环：第 {i + 1} 次迭代" + "="*8)
    # generate
    if i == 0:
        print(f">>> 阶段 1: 生成初始代码...")
    else:
        print(f">>> 阶段 1: 根据批判优化代码...")
        message_history.append(HumanMessage(content="根据批判意见优化代码。"))
    response = llm.invoke(message_history)
    message_history.append(response)
    current_code = response.content
    print(f">>> 生成代码（第{i+1}版）：")
    print(current_code)
    print()
    # reflection
    print(">>> 阶段 2: 对生成代码进行反思...")
    critique_response = llm.invoke([
        SystemMessage(content=reflect_prompt),
        HumanMessage(content=f"原始任务：\n{task_prompt}\n\n 带审查代码：\n{current_code}")
    ])
    critique = critique_response.content
    # examine
    if "CODE_IS_PERFECT" in critique:
        print("代码已达要求。")
        break
    print(f">>> 审查意见：")
    print(critique)
    print()
    message_history.append(HumanMessage(content=f"上次代码批判意见：{critique}"))

========反思循环：第 1 次迭代========
>>> 阶段 1: 生成初始代码...
>>> 生成代码（第1版）：
以下是根据您的要求编写的 `calculate_factorial` 函数：

```python
def calculate_factorial(n):
    """
    Calculate the factorial of a non-negative integer n.

    The factorial of n (denoted as n!) is the product of all positive integers
    less than or equal to n. By convention, 0! = 1.

    Parameters:
    n (int): A non-negative integer whose factorial is to be computed.

    Returns:
    int: The factorial of n.

    Raises:
    TypeError: If n is not an integer.
    ValueError: If n is negative.
    """
    # 验证输入类型
    if not isinstance(n, int):
        raise TypeError("Input must be an integer.")
    # 处理负数情况
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    
    # 计算阶乘（迭代法）
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

### 使用说明
- **正常情况**：传入非负整数，返回其阶乘。例如 `calculate_factorial(5)` 返回 `120`。
- **边界情况**：`calculate_factorial(0)` 返回 `1`。
- **无效输入**：
 

In [ ]:
def run_reflection(task_prompt: str, reflect_prompt: str, success_key: str,
                   max_step: int = 3) -> tuple[int, bool, str]:
    message_history = [HumanMessage(content=task_prompt)]
    generated = ""
    is_suc = False
    for i in range(max_step):
        print("="*8 + f"Task {i + 1}th Iteration" + "="*8)
        # generate
        if i == 0:
            print(f">>> Phase 1: Generate content...")
        else:
            print(f">>> Phase 1: Optimize content due to critique...")
            message_history.append(HumanMessage(content="Optimize due to critique."))
        response = llm.invoke(message_history)
        message_history.append(response)
        generated = response.content
        print(f">>> Generated (V{i+1}):")
        print(generated)
        print()
        # reflection
        print(">>> Phase 2: Reflect generated content...")
        critique_response = llm.invoke([
            SystemMessage(content=reflect_prompt),
            HumanMessage(
                content=f"Origin task:\n{task_prompt}\n\n Review content:\n{generated}")
        ])
        critique = critique_response.content
        # examine
        if success_key in critique:
            print("Success!")
            is_suc = True
            break
        print(f">>> Critique:")
        print(critique)
        print()
        message_history.append(HumanMessage(content=f"Last critique: {critique}"))
    return (i+1, is_suc, generated)

In [10]:
step, is_succeed, code = run_reflection(
    task_prompt, reflect_prompt, success_key)

print(f"Run {step} rounds, result: {is_succeed}")
print("Final generated:")
print(code)

========Task 1th Iteration========
>>> Phase 1: Generate content...
>>> Generated (V1):
下面是一个满足要求的 `calculate_factorial` 函数实现：

```python
def calculate_factorial(n: int) -> int:
    """
    计算非负整数 n 的阶乘 (n!)。

    阶乘定义为 n! = n * (n-1) * (n-2) * ... * 1，
    其中 0! = 1。

    参数:
        n (int): 要计算阶乘的非负整数。

    返回:
        int: n 的阶乘值。

    抛出:
        ValueError: 如果 n 为负数。

    示例:
        >>> calculate_factorial(5)
        120
        >>> calculate_factorial(0)
        1
        >>> calculate_factorial(3)
        6
    """
    if not isinstance(n, int):
        raise TypeError("输入必须为整数")
    if n < 0:
        raise ValueError("不能计算负数的阶乘")
    
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

**说明：**
- 函数首先检查输入是否为整数（可选，但提升健壮性），然后检查是否为负数，若是则抛出 `ValueError`。
- 使用循环从 2 乘到 n（当 n 为 0 或 1 时，循环不执行，结果保持为 1）。
- 包含了清晰的 docstring，说明功能、参数、返回值、异常以及示例。
- 边界情况 `n=0` 直接返回 1，正确。

>>> Phase 2: Reflect generated content...
Success!
Run 1 rounds, result: True
Final g